# Execution test — the checkers decide each check in the plan

For each DEV case:

1. **The plan** is read back from the parser's cache on Drive: exactly the plan the frozen parser (Qwen3.8-27B, fix4) wrote in `Parser_Test`. Nothing is parsed again and no local model is loaded. If a plan is not in the cache, the notebook stops and says so.
2. **The checkers** decide each check, in the order the plan allows (Eq 4):
   - **calculator** and **rule checker**: exact tools, no model;
   - **cross-reference checker**: does the cited section, table or figure exist;
   - **text checker** and **image checker**: **Claude Sonnet 5.5** through the Anthropic API.

   A calculator or rule check that cannot decide goes to the text checker (§3.3). Every text and image check first gets the evidence the plan points to (its source paragraph and its hints), then what a search on the claim adds (Eq 5).
3. **The merges and the decision** are worked out by the executor, with no model. The answer here is written from the certificates, with no model.

**Repeatable by caching.** Sonnet 5.5 does not accept a temperature setting, so every checker reply is saved on Drive (`parser_runs/checker_cache/`) with its prompt, images, reply, stop reason and token usage. A re-run reads the saved replies. CELL 10 measures how much a fresh run would differ.

**Only DEV cases** (written from random MUTCD provisions). No test question, gold answer or test plan is in this notebook.

**Before you start:** put your Anthropic API key in Colab's secrets (the key icon on the left) as `ANTHROPIC_API_KEY`, and allow this notebook to use it.

**Runtime: a GPU (A100 or L4)** for the retrieval models (search encoder and reranker). No model server is started.

**Run CELL 1 to CELL 8, then stop and send me the output of CELL 8.** CELL 9 runs all 11 DEV cases; CELL 10 checks repeatability.

In [ ]:
# CELL 1 — Drive, repo, packages.
import sys, os, subprocess
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/evaluation")

# The retrieval stack, pinned as in Retrieval_Test and Parser_Test.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"

for f in ("mrag/vine/pointers.py", "mrag/vine/anthropic_client.py", "evaluation/execution_check.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"{f} is not in the clone -- push it to GitHub, then re-run this cell"
assert "NOT_APPLICABLE" in open(f"{REPO_DIR}/mrag/vine/certificate.py").read(), (
    "mrag/vine/certificate.py is the old version -- push the executor change first")

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("data  :", DRIVE_DIR)
print("code  :", REPO_DIR, "| commit", COMMIT)

In [ ]:
# CELL 2 — the vector store on local disk (restored from Drive if needed).
import shutil, time
from pathlib import Path
from mrag.config import CFG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = Path(CFG.base_dir) / "qdrant_db.tar"

def has_store(p: Path) -> bool:
    return (p / "collection" / CFG.coll_chunks).exists()

if has_store(LOCAL):
    print("vector store already on local disk:", LOCAL)
elif SNAP.exists():
    print(f"restoring {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) ...")
    t0 = time.time()
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)
    print(f"  done in {time.time() - t0:.0f}s")
else:
    raise FileNotFoundError(f"No vector store at {LOCAL} and no snapshot at {SNAP}. "
                            "Run notebooks/Fresh_Build.ipynb first.")
assert has_store(LOCAL), f"{LOCAL}/collection/{CFG.coll_chunks} is missing after the restore"
print("ok:", LOCAL)

In [ ]:
# CELL 3 — the retriever the checkers use during execution (Eq 5), and the tables.
# The same retrieval stack as Parser_Test (text encoder, reranker, VINE graph). No LLM.
import logging, json, torch
logging.basicConfig(level=logging.WARNING)
from mrag.ask import init_pipeline
from mrag.vine import table_data

pipeline = init_pipeline(load_image_embedder=False, load_vlm=False)
assert type(pipeline.kg).__name__ == "VineKG", "retrieval is not on the VINE graph -- check CFG.graph_backend"
retriever, kg = pipeline.retriever, pipeline.kg

TABLES = {}
for t in table_data.load(Path(CFG.tables_jsonl)):
    TABLES.setdefault(t.table_id, []).append(t)
TABLE_LIST = [t for ts in TABLES.values() for t in ts]

print(f"graph   : {kg.g.number_of_nodes():,} nodes, {kg.g.number_of_edges():,} edges")
print(f"tables  : {len(TABLES)} table ids")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (retrieval will be slow)'}")

In [ ]:
# CELL 4 — the DEV cases, what retrieval gave the parser for them, and the frozen
# parser's settings (read from the Parser_Test run that made the DEV plans).
import importlib, hashlib
import parser_check as pc
import execution_check as ec
from mrag.vine.parser import PARSER_TASK, PARSER_CONTRACT
importlib.reload(pc); importlib.reload(ec)

DEV = pc.load_dev_cases()
RUNS = Path(CFG.base_dir) / "parser_runs"
RETRIEVAL_FILES = ["mrag/retrieval.py", "mrag/kg_vine.py", "mrag/kg.py", "mrag/config.py",
                   "mrag/embeddings.py", "mrag/vector_store.py", "mrag/question_router.py",
                   "mrag/vine/graph_links.py", "mrag/vine/graph_enrich.py", "mrag/vine/vector_items.py"]
RETR_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h", "--"] + RETRIEVAL_FILES,
                             capture_output=True, text=True).stdout.strip()
_manifest = Path(CFG.base_dir) / "vine_data" / "text_store_manifest.json"
STORE_TAG = hashlib.sha256(_manifest.read_bytes()).hexdigest()[:6] if _manifest.exists() else "nostore"
KQ_FILE = RUNS / f"kq_r{RETR_COMMIT}_s{STORE_TAG}.json"
assert KQ_FILE.exists(), (f"{KQ_FILE.name} is not on Drive. Run Parser_Test CELL 1-5 first: the plans "
                          "here must be made from the same retrieval results.")
KQ = json.loads(KQ_FILE.read_text())
missing = [c["case_id"] for c in DEV if c["case_id"] not in KQ]
assert not missing, f"not retrieved yet: {missing} -- run Parser_Test CELL 5"

INSTRUCTIONS_SHA = hashlib.sha256((PARSER_TASK + PARSER_CONTRACT).encode()).hexdigest()
parser_runs = []
for mf in sorted(RUNS.glob("run_*/manifest.json")):
    m = json.loads(mf.read_text())
    if m.get("instructions_sha256") == INSTRUCTIONS_SHA and m.get("reasoning_effort") == "medium":
        parser_runs.append((mf.parent.name, m))
assert parser_runs, ("no Parser_Test run with the current (frozen) parser instructions was found on "
                     "Drive -- run Parser_Test CELL 1-10 first")
PARSER_RUN, PM = parser_runs[-1]
print(len(DEV), "DEV cases; retrieval results from", KQ_FILE.name)
print(f"parser  : {PM['model']} rev {PM['revision'][:12]} | {PM['vllm']} | effort {PM['reasoning_effort']} "
      f"| max_tokens {PM['max_tokens']} | settings from {PARSER_RUN}")

In [ ]:
# CELL 5 — the checker model: Claude Sonnet 5.5 through the API, every reply saved.
from mrag.vine import anthropic_client as ac
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception as e:
    assert os.environ.get("ANTHROPIC_API_KEY"), (
        "Add ANTHROPIC_API_KEY to Colab's secrets (key icon on the left), allow this notebook "
        f"to use it, then re-run this cell. ({e!r})")

CHECKER_MODEL, CHECKER_EFFORT, CHECKER_MAX_TOKENS = "claude-sonnet-5-5", "high", 32000
MAX_IMAGES = 4
CHECKER_CACHE = RUNS / "checker_cache"
CHECKER_ASK = ac.make_ask_anthropic(CHECKER_MODEL, effort=CHECKER_EFFORT, max_tokens=CHECKER_MAX_TOKENS,
                                    cache_dir=str(CHECKER_CACHE))

reply = CHECKER_ASK('Reply with this JSON object and nothing else: {"ok": true}', [])
print("text probe :", repr(reply[:80]), "|", CHECKER_ASK.last["finish_reason"], "|", CHECKER_ASK.last["model"])
assert '"ok"' in reply, "the API answered, but not with the JSON asked for -- send the output above"

fig = kg.figure("Figure 2B-1")
assert fig, "Figure 2B-1 is not in the graph -- send the output above"
img = kg.g.nodes[fig].get("image_path") or (kg.g.nodes[fig].get("image_paths") or [""])[0]
assert os.path.exists(img), f"the figure image is not on Drive: {img}"
reply = CHECKER_ASK('Reply with one JSON object and nothing else: {"shape": "<the shape of the first '
                    'sign shown, in one or two words>"}', [img])
print("image probe:", repr(reply[:120]), "|", CHECKER_ASK.last["finish_reason"], "|", img)
print(f"cost so far: ${ac.cost_usd(CHECKER_ASK.calls, CHECKER_MODEL):.4f}")

In [ ]:
# CELL 6 — the DEV plans, read back from the parser's cache. No model is called:
# the same settings as the Parser_Test run, so each plan is the one the frozen
# parser (fix4) wrote. cache_only=True: a plan that is not saved stops this cell.
from mrag.vine import vllm_client as vc
from mrag.vine.parser import make_semantic_parser
from mrag.vine.compile import instantiate

PARSER_ASK = vc.make_ask_vllm("http://unused", "parser", model_id=PM["model"], revision=PM["revision"],
                              engine=f"vllm {PM['vllm']}", seed=PM["seed"], max_tokens=PM["max_tokens"],
                              sampling=PM["sampling"], reasoning_effort=PM["reasoning_effort"],
                              cache_dir=str(RUNS / "model_cache"), cache_only=True)
parse = make_semantic_parser(PARSER_ASK, max_attempts=3, fall_back_to_baseline=False,
                             check_ref=kg.is_known_citation, tables=TABLES)
PLANS = {}
for c in DEV:
    v = KQ[c["case_id"]]
    spec, rep = parse(v["question"], v["chunks"], "", v["figures"])
    gaps = [p for group in rep.problems for p in group if "NotInCache" in p]
    assert not gaps, (f"{c['case_id']}: its plan is not in the parser cache. Run Parser_Test CELL 10 "
                      "(DEV, medium) first, then this cell.")
    net, problems = instantiate(spec) if spec is not None else (None, ["no plan"])
    PLANS[c["case_id"]] = {"spec": spec, "net": net, "problems": problems}
    kinds = {}
    for o in (net.operations if net is not None else []):
        if o.merge is None:
            kinds[o.verifier] = kinds.get(o.verifier, 0) + 1
    print(f"  {c['case_id']:26s} plan {'ok' if net is not None and not problems else 'REJECTED':8s} "
          f"checks by checker {kinds}")

In [ ]:
# CELL 7 — the checkers, and one function that executes a plan.
from mrag.vine.run import build_verifiers
from mrag.vine.execute import execute
from mrag.vine.answer import compose

MODEL_OPTIONS = {"use_pointers": True, "max_pointed_chunks": 8, "max_images": MAX_IMAGES}
RUN_DIR = RUNS / time.strftime("exec_%Y%m%d_%H%M")
RUN_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = {"commit": COMMIT, "parser_run": PARSER_RUN, "parser_model": PM["model"],
            "parser_revision": PM["revision"], "instructions_sha256": INSTRUCTIONS_SHA,
            "checker_model": CHECKER_MODEL, "checker_effort": CHECKER_EFFORT,
            "checker_max_tokens": CHECKER_MAX_TOKENS, "checker_temperature": "not settable (model default)",
            "model_options": MODEL_OPTIONS, "hand_over": True, "kq_file": KQ_FILE.name}
(RUN_DIR / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))

def run_case(case, ask=None):
    ask = ask or CHECKER_ASK
    cid, q = case["case_id"], case["text"]
    plan = PLANS[cid]
    n0 = len(ask.calls)
    t0 = time.time()
    if plan["net"] is None or plan["problems"]:
        return ec.make_exec_record(cid, q, plan["spec"], plan["net"], None, None, [], 0.0, True)
    verifiers = build_verifiers(tables=TABLE_LIST, kg=kg, retriever=retriever, ask=ask,
                                query=q, hand_over=True, model_options=MODEL_OPTIONS)
    trace = execute(plan["net"], verifiers)
    answer = compose(q, plan["net"], trace)
    rec = ec.make_exec_record(cid, q, plan["spec"], plan["net"], trace, answer,
                              ask.calls[n0:], time.time() - t0, True)
    rec["cost_usd"] = ac.cost_usd(ask.calls[n0:], CHECKER_MODEL)
    return rec

print("checker:", CHECKER_MODEL, "| effort", CHECKER_EFFORT, "| run folder", RUN_DIR)

### CELL 8 executes one DEV case. Read it before going on.

Things to look for:
- **each check's status** and **who decided it** (`calculator -> llm` means the calculator could not decide and handed it to the text checker);
- **cited** — the evidence the checker says it used; **pointed** — what the plan pointed to. A text or image check that cites nothing it was given is turned into UNKNOWN;
- **reason** — the checker's own one-line reason;
- **merges** — their inputs, and the result the executor worked out;
- **decision** — and whether it is certified.

Send me the output.

In [ ]:
# CELL 8 — one DEV case.
first = DEV[0]
REC0 = run_case(first)
ec.print_execution(REC0)
print(f"cost: ${REC0.get('cost_usd', 0):.4f}")
(RUN_DIR / f"{first['case_id']}.json").write_text(json.dumps(REC0, indent=1, default=str))

In [ ]:
# CELL 9 — every DEV case, then the summary.
EXEC_RECORDS = []
for i, c in enumerate(DEV, 1):
    r = run_case(c)
    EXEC_RECORDS.append(r)
    print(f"  {i:2d}/{len(DEV)} {c['case_id']:26s} decision {str(r['terminal']):15s} "
          f"checks {sum(1 for x in r['certificates'] if x['kind'] == 'check'):2d} "
          f"model calls {len(r['calls']):2d} | {r['seconds']:5.0f}s | ${r.get('cost_usd', 0):.3f}")
    (RUN_DIR / "exec_dev.json").write_text(json.dumps(EXEC_RECORDS, indent=1, default=str))
print()
ec.print_exec_summary(ec.summarize_execution(EXEC_RECORDS))
print(f"cost of new calls: ${sum(r.get('cost_usd', 0) for r in EXEC_RECORDS):.3f}")
print("\nsaved ->", RUN_DIR / "exec_dev.json", "\n")
for r in EXEC_RECORDS:
    ec.print_execution(r)
    print()

In [ ]:
# CELL 10 — repeatability: three DEV cases again, asking the checker afresh
# (no saved replies). Counts how many checks come out differently. This is the
# number to report, since the checker's temperature cannot be set.
fresh = ac.make_ask_anthropic(CHECKER_MODEL, effort=CHECKER_EFFORT, max_tokens=CHECKER_MAX_TOKENS,
                              cache_dir=None)
same = differ = 0
REPEAT = []
for r in EXEC_RECORDS[:3]:
    case = next(c for c in DEV if c["case_id"] == r["case_id"])
    again = run_case(case, ask=fresh)
    REPEAT.append(again)
    before = {x["id"]: x["status"] for x in r["certificates"]}
    after = {x["id"]: x["status"] for x in again["certificates"]}
    flips = [(k, before[k], after.get(k)) for k in before if before[k] != after.get(k)]
    same += len(before) - len(flips)
    differ += len(flips)
    print(f"  {r['case_id']:26s} decision {r['terminal']} -> {again['terminal']} | "
          f"{len(flips)} of {len(before)} results differ {flips}")
print(f"\n  {differ} of {same + differ} results differ on a fresh run | cost ${sum(a.get('cost_usd', 0) for a in REPEAT):.3f}")
MANIFEST["repeat_check"] = f"{differ} of {same + differ} results differ on a fresh run (3 DEV cases)"
(RUN_DIR / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))
(RUN_DIR / "exec_dev_repeat.json").write_text(json.dumps(REPEAT, indent=1, default=str))